# Test whether a SQL query returns real rows from the database
## Run each cell in order to validate a query against the live Supabase/PostgreSQL connection.
This notebook is meant for quick DB checks. Replace the sample query in Cell 4 with any SELECT query you want to test.

In [6]:
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus, urlparse

import pandas as pd
import psycopg


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".env").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing .env and src/.")


ROOT = find_project_root(Path.cwd().resolve())
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ENV_PATH = ROOT / ".env"


def load_database_url() -> str:
    env_url = os.getenv("SUPABASE_DATABASE_URL")
    if env_url:
        return env_url

    if not ENV_PATH.exists():
        raise FileNotFoundError("No .env file found in the project root.")

    values = {}
    for raw_line in ENV_PATH.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")

    if values.get("SUPABASE_DATABASE_URL"):
        return values["SUPABASE_DATABASE_URL"]

    host = values.get("SUPABASE_DB_HOST") or ""
    if not host:
        project_url = values.get("NEXT_PUBLIC_SUPABASE_URL")
        if project_url:
            parsed = urlparse(project_url)
            host = parsed.hostname or ""
            if host and not host.startswith("db."):
                host = f"db.{host}"

    password = values.get("SUPABASE_DB_PASSWORD")
    if not host or not password:
        raise ValueError("Set SUPABASE_DATABASE_URL or SUPABASE_DB_HOST + SUPABASE_DB_PASSWORD in .env.")

    user = values.get("SUPABASE_DB_USER") or "postgres"
    dbname = values.get("SUPABASE_DB_NAME") or "postgres"
    port = values.get("SUPABASE_DB_PORT") or "5432"
    return f"postgresql://{user}:{quote_plus(password)}@{host}:{port}/{dbname}?sslmode=require"


DATABASE_URL = load_database_url()
print("Database URL loaded successfully.")
print("Using host:", urlparse(DATABASE_URL).hostname)


Database URL loaded successfully.
Using host: aws-1-ap-northeast-1.pooler.supabase.com


In [8]:
SQL_QUERY = """
select * from departments;
""".strip()

print("Query to test:")
print(SQL_QUERY)


Query to test:
select * from departments;


In [9]:
import sys
from pathlib import Path

import psycopg


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".env").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing .env and src/.")


ROOT = find_project_root(Path.cwd().resolve())
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from nl2sql.validation import SQLValidationGatekeeper

safety = SQLValidationGatekeeper(database_url=None).validate(SQL_QUERY)
print("Safety status:", safety.status.value)
print("Safety message:", safety.user_message)
if not safety.ok:
    raise RuntimeError(f"Query safety check failed: {safety.user_message}")

try:
    with psycopg.connect(DATABASE_URL) as conn:
        with conn.cursor() as cur:
            cur.execute(SQL_QUERY)
            rows = cur.fetchall()
            columns = [desc[0] for desc in cur.description]
except Exception as exc:
    rows = []
    columns = []

    print("DB execution failed:", exc)

print("Columns:", columns)
print("Rows returned:", len(rows))

Safety status: ok
Safety message: SQL passed safety checks.
Columns: ['department_id', 'department_name', 'location', 'phone', 'head_of_department']
Rows returned: 20


In [10]:
df = pd.DataFrame(rows, columns=columns)
df


,department_id,department_name,location,phone,head_of_department
0,1,Cardiology,"Building A, Floor 3",+94711419610,Dr. Waruna Abeysekara
1,2,Neurology,"Building A, Floor 4",+94715108603,Dr. Niroshan Peiris
2,3,Orthopedics,"Building B, Floor 2",+94712719583,Dr. Faisal Kumarasiri
3,4,Pediatrics,"Building C, Floor 1",+94712458591,Dr. Esandu Padmasiri
4,5,Oncology,"Building A, Floor 5",+94711533224,Dr. Pathum Yapa
5,6,Dermatology,"Building D, Floor 1",+94714668136,Dr. Dinesh Wijesinghe
6,7,Gastroenterology,"Building B, Floor 3",+94711445199,Dr. Gimhan Jeyaraj
7,8,Pulmonology,"Building A, Floor 6",+94718038374,Dr. Kavinda Gunawardana
8,9,Nephrology,"Building B, Floor 4",+94715667265,Dr. Faisal Balasuriya
9,10,Endocrinology,"Building C, Floor 2",+94718090293,Dr. Ashan Gamage


In [5]:
if df.empty:
    print("No rows were returned. Check the database connection or change the query.")
else:
    print("Real DB result confirmed: this query returned rows from the database.")


Real DB result confirmed: this query returned rows from the database.
